# Task #30 (Story #5): Khảo sát dữ liệu thiếu ngày giao thực tế

Mục tiêu: đọc `data/processed/orders_joined.csv`, ép kiểu lại các cột thời gian (CSV không giữ dtype datetime), và thống kê phân bố đơn thiếu `order_delivered_customer_date` theo `order_status` để làm căn cứ chọn phương án xử lý ở Task #31.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_joined.csv")
print(df.shape)
df.head(2)

(99441, 41)


C:\Users\thanh\AppData\Local\Temp\ipykernel_25448\3542643689.py:3: DtypeWarning: Columns (0: payment_has_boleto, 1: payment_has_credit_card, 2: payment_has_debit_card, 3: payment_has_not_defined, 4: payment_has_voucher) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/orders_joined.csv")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,items_num_items,items_num_products,...,primary_seller_id,primary_seller_zip_code_prefix,primary_seller_city,primary_seller_state,items_multi_seller,items_num_categories,review_score_avg,review_score_min,review_score_max,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,1.0,...,3504c0cb71d7fa48d967e0e4c94d59d9,9350.0,maua,SP,False,1.0,4.0,4.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,1.0,...,289cdb325fb7e7f891c38608bf9e0962,31570.0,belo horizonte,SP,False,1.0,4.0,4.0,4.0,1.0


## Ép kiểu lại cột ngày tháng

CSV không giữ dtype datetime qua vòng đọc/ghi — 5 cột thời gian của `orders` đọc lên đều là `object` (chuỗi), phải `pd.to_datetime` lại.

In [2]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print(df[date_cols].dtypes)
for col in date_cols:
    df[col] = pd.to_datetime(df[col])
print("---after cast---")
print(df[date_cols].dtypes)

order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
---after cast---
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Thống kê đơn thiếu `order_delivered_customer_date`

In [3]:
missing_mask = df["order_delivered_customer_date"].isna()
n_missing = missing_mask.sum()
n_total = len(df)
print(f"Thiếu order_delivered_customer_date: {n_missing}/{n_total} ({n_missing / n_total:.2%})")

Thiếu order_delivered_customer_date: 2965/99441 (2.98%)


## Phân bố theo `order_status`

Kiểm tra xem đơn thiếu ngày giao có chủ yếu rơi vào các status như `canceled`/`unavailable`/`processing`, hay có cả đơn `delivered` bị thiếu (dữ liệu bất thường).

In [4]:
status_breakdown = (
    df.assign(missing_delivered=missing_mask)
    .groupby("order_status")["missing_delivered"]
    .agg(n_orders="count", n_missing="sum")
)
status_breakdown["pct_missing"] = (status_breakdown["n_missing"] / status_breakdown["n_orders"]).map("{:.2%}".format)
status_breakdown

,n_orders,n_missing,pct_missing
order_status,,,
approved,2,2,100.00%
canceled,625,619,99.04%
created,5,5,100.00%
delivered,96478,8,0.01%
invoiced,314,314,100.00%
processing,301,301,100.00%
shipped,1107,1107,100.00%
unavailable,609,609,100.00%


In [5]:
# Trong số các đơn thiếu ngày giao, status nào chiếm bao nhiêu %
df.loc[missing_mask, "order_status"].value_counts(normalize=True).map("{:.2%}".format)

order_status
shipped        37.34%
canceled       20.88%
unavailable    20.54%
invoiced       10.59%
processing     10.15%
delivered       0.27%
created         0.17%
approved        0.07%
Name: proportion, dtype: str